# FF49 — Neural CDE diffusion

## A. Experiment and configuration

Daily value-weighted Kenneth French returns, **49 assets**, arithmetic decimal
returns minus aligned daily French RF, train-only Z-scores, lookback 60, horizon 1.
The model is the existing conditional Gaussian diffusion with CDE history
conditioning, cosine schedule, v-prediction and MSE; optimizer, seed 42, solver
and all model settings come from `configs/cde_ff49.yaml` without notebook overrides.

Requested sample: **1958-01-01–2023-12-31**. Train targets end **1999-12-31**;
validation targets are **2000-01-01–2004-12-31**; the held-out period after that is
never evaluated here. Complete-case exclusions change actual coverage. No imputation
is permitted, and histories/targets cannot cross gaps exceeding seven calendar days.

Principal high-dimensional forecasting comparison. The complete panel starts on 1969-07-01, with one excluded source date (1971-03-11) afterwards. The seven-day guard is coarse and does not detect every isolated missing trading day. Cached 202608 data yield 7,648 training, 1,256 validation and 4,781 test windows.

`smoke` is the default and performs no optimizer updates. Opt into `pilot` for two
epochs first, then `full` for the unchanged 50-epoch configuration. Pilot and full
start independently from the configured seed in separate directories; pilot weights
are never copied into full. FF49 Vanilla/CDE share data, seeds and evaluation settings;
do not compare FF49/KF12 as matched-coverage experiments. No findings are assumed.

Artifacts include a smoke JSON/log, semantic test log, raw archive snapshots/hashes,
Git/config provenance, epoch histories, last/best checkpoints, session timing/resources,
and (only after full completion) validation scenarios, metrics, calibration and curves.


In [ ]:
MODE = "smoke"                     # "smoke", "pilot", or "full"; training is opt-in
BRANCH = "main"                    # Before merge, select the feature branch containing these files
REVISION = None                    # Pin the saved Git SHA when resuming after main advances
RUN_LABEL = None                   # Defaults to seed42; use a new label for a distinct experiment
PILOT_EPOCHS = 2                   # Absolute stop epoch, not additional epochs on each resume
FULL_STOP_AFTER_EPOCH = None        # Optional absolute chunk boundary; configuration stays at 50
DEVICE = "auto"                    # Smoke can use CPU; pilot/full require a Kaggle GPU
EVAL_BATCH_SIZE = 32                # Matched across the paired Vanilla/CDE evaluations
RESTORE_ARTIFACTS = True
UPLOAD_ARTIFACTS = False            # Explicit opt-in; Kaggle credentials must permit dataset uploads
UPLOAD_STAGE = "pilot"             # "pilot" or "full"; independent of MODE for intermediate backups

CONFIG_PATH = "configs/cde_ff49.yaml"
DATA_REFERENCE = "configs/cde_ff49.yaml"
EXPERIMENT = "ff49/neural_cde"
EXPECTED_DATASET, EXPECTED_ENCODER, EXPECTED_ASSETS = "ken_french_49", "cde", 49
RETURN_ARCHIVE = "49_Industry_Portfolios_daily_CSV.zip"
SMOKE_PASSED = False


## B. Kaggle setup and restoration

Enable **Internet**. Select a **GPU accelerator** for pilot/full; CPU smoke is supported.
The setup force-syncs only the disposable `/kaggle/working/diffusion-portfolio` checkout,
as in the KF12 notebooks. Run outputs live outside that checkout. Before this feature
merges, change `BRANCH` in the configuration cell to its feature branch; otherwise
keep `main`. The current commit is printed and saved. A kernel restart is appropriate
when changing revisions after importing the package.

The existing `dev,cde` extras supply tests and CDE support; no topology dependency
is needed. Kaggle normally supplies `kagglehub`. Restore requires read access;
uploads additionally require your authenticated Kaggle account's write access.
Only this dataset/model/seed namespace is copied locally. No artifact-store marker
is required. Existing local outputs take precedence and are never force-restored.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_PATH = Path("/kaggle/working/diffusion-portfolio")
repo_url = "https://github.com/Amir-Entezari/diffusion-portfolio.git"
if not (REPO_PATH / ".git").exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", repo_url, str(REPO_PATH)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_PATH), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_PATH), "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run(["git", "-C", str(REPO_PATH), "reset", "--hard", "FETCH_HEAD"], check=True)
if REVISION is not None:
    subprocess.run(["git", "-C", str(REPO_PATH), "checkout", "--detach", REVISION], check=True)
os.chdir(REPO_PATH)
sys.path.insert(0, str(REPO_PATH / "src"))
GIT_SHA = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Repository:", REPO_PATH, "branch:", BRANCH, "commit:", GIT_SHA)


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev,cde]"], check=True)

import json
import time
import hashlib
import signal
import shutil
import tempfile
from datetime import datetime, timezone
from dataclasses import asdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from diffusion_portfolio.config import load_config
from diffusion_portfolio.data.preparation import load_standardizer
from diffusion_portfolio.utils.device import resolve_device

env = os.environ.copy()
env.update(PYTHONUNBUFFERED="1", OMP_NUM_THREADS="1")
device = resolve_device(DEVICE)
GPU_NAME = torch.cuda.get_device_name(device) if device.type == "cuda" else None
print("Torch:", torch.__version__, "CUDA available:", torch.cuda.is_available(), "GPU:", GPU_NAME)
if MODE != "smoke" and device.type != "cuda":
    raise RuntimeError("Enable a Kaggle GPU before opting into pilot or full training.")

def sha256(path):
    with Path(path).open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def run_logged(command, log_path):
    # Fail on subprocess errors while retaining and displaying the real output.
    with Path(log_path).open("w") as log:
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1, env=env, start_new_session=True) as process:
            try:
                for line in process.stdout:
                    print(line, end="")
                    log.write(line)
                    log.flush()
            except BaseException:
                # Stop the whole CLI group, including a GNU-time wrapper, before backup.
                try:
                    os.killpg(process.pid, signal.SIGTERM)
                except ProcessLookupError:
                    pass
                process.wait()
                raise
            if process.wait() != 0:
                raise subprocess.CalledProcessError(process.returncode, command)


In [ ]:
assert MODE in {"smoke", "pilot", "full"}, "Choose smoke, pilot or full"
if not Path(CONFIG_PATH).is_file():
    raise FileNotFoundError(f"{CONFIG_PATH}: before merge, set BRANCH to the feature branch containing this config.")
cfg = load_config(CONFIG_PATH)
reference = load_config(DATA_REFERENCE)
assert cfg.data == reference.data and cfg.seed == reference.seed, "Dataset/seed must match the paired experiment"
assert cfg.model.history_encoder == EXPECTED_ENCODER
assert cfg.data.dataset == EXPECTED_DATASET
assert cfg.data.subtract_risk_free and cfg.data.source_options == {"missing_policy": "drop"}
assert cfg.data.max_gap_days == 7 and cfg.training.epochs == 50
assert type(PILOT_EPOCHS) is int and 0 < PILOT_EPOCHS < cfg.training.epochs
assert FULL_STOP_AFTER_EPOCH is None or (type(FULL_STOP_AFTER_EPOCH) is int and 0 < FULL_STOP_AFTER_EPOCH <= cfg.training.epochs)
label = RUN_LABEL or f"seed{cfg.seed}"
assert label and not Path(label).is_absolute() and len(Path(label).parts) == 1 and label not in {".", ".."}
RUN_ROOT = Path("/kaggle/working/high_dimensional") / EXPERIMENT / label
ARTIFACT_NAMESPACE = f"high_dimensional/{EXPERIMENT}/{label}"
ARTIFACT_DATASET = "amirentezari/diffusion-portfolio"
PILOT_DIR, FULL_DIR, SMOKE_DIR = (RUN_ROOT / name for name in ("pilot", "full", "smoke"))
CONFIG = json.loads(json.dumps(asdict(cfg)))
ARCHIVES = [Path("data/raw") / RETURN_ARCHIVE, Path("data/raw/F-F_Research_Data_Factors_daily_CSV.zip")]
print("Config:", CONFIG_PATH, "seed:", cfg.seed, "encoder:", cfg.model.history_encoder)
print("Run root:", RUN_ROOT, "artifact namespace:", ARTIFACT_NAMESPACE)


In [ ]:
import kagglehub

# Copy only this experiment. Existing local stages take precedence, while a
# smoke-only local directory does not prevent restoration of pilot/full stages.
if RESTORE_ARTIFACTS:
    try:
        artifact_path = Path(kagglehub.dataset_download(ARTIFACT_DATASET))
    except Exception as error:
        active = PILOT_DIR if MODE == "pilot" else FULL_DIR
        if MODE != "smoke" and not (active / "last.pt").exists():
            raise RuntimeError("Restore failed. Resolve access before training, or explicitly disable restoration for a new run.") from error
        print("Restore unavailable; keeping local artifacts:", error)
    else:
        source = artifact_path / ARTIFACT_NAMESPACE
        if source.is_dir():
            remote_provenance = source / "provenance.json"
            if not remote_provenance.is_file():
                raise RuntimeError("Saved experiment lacks provenance; refusing to adopt it.")
            local_provenance = RUN_ROOT / "provenance.json"
            if local_provenance.exists():
                assert json.loads(local_provenance.read_text()) == json.loads(remote_provenance.read_text()), "Remote/local experiment provenance mismatch"
            RUN_ROOT.mkdir(parents=True, exist_ok=True)
            for item in source.iterdir():
                destination = RUN_ROOT / item.name
                if destination.exists():
                    print("Keeping local:", destination)
                elif item.is_dir():
                    shutil.copytree(item, destination)
                else:
                    shutil.copy2(item, destination)
            print("Restored missing artifacts from:", source)
        else:
            print("No saved experiment at:", ARTIFACT_NAMESPACE)

# Rehydrate exact source ZIPs before data loading. Conflicting caches fail.
for archive in ARCHIVES:
    snapshot = RUN_ROOT / "inputs" / archive.name
    if snapshot.exists():
        if archive.exists() and sha256(archive) != sha256(snapshot):
            raise RuntimeError(f"Source archive mismatch: {archive}; resolve the cache before resuming.")
        archive.parent.mkdir(parents=True, exist_ok=True)
        if not archive.exists():
            shutil.copy2(snapshot, archive)


## C. Prerequisites and smoke validation

Run the existing temporal-leakage, config-wiring and continuity tests, then
`scripts/data_smoke_test.py --model-smoke`. The script uses shared preparation,
derives the asset count, checks shapes/inverse scaling/chronology, and checks one
training batch's finite forward/loss/backward. Test window counts are metadata
only: no test forecasts or metrics are computed. A subprocess failure stops this
cell, and the training guard stays false. Counts below are from execution, not
hard-coded scientific results. Raw data and RF ZIPs are saved to reproduce a
later session even if the French library revises its files.


In [ ]:
SMOKE_PASSED = False
SMOKE_DIR.mkdir(parents=True, exist_ok=True)
report_path = SMOKE_DIR / "report.json"
report_path.unlink(missing_ok=True)
run_logged([sys.executable, "-m", "pytest", "-q",
            "tests/semantic/test_data_leakage.py", "tests/semantic/test_window_continuity.py",
            "tests/semantic/test_config_wiring.py"], SMOKE_DIR / "semantic_tests.log")
run_logged([sys.executable, "scripts/data_smoke_test.py", "--config", CONFIG_PATH,
            "--model-smoke", "--smoke-batch-size", "1", "--device", str(device),
            "--report-json", str(report_path)], SMOKE_DIR / "data_smoke.log")
SMOKE = json.loads(report_path.read_text())
assert SMOKE["dataset"] == EXPECTED_DATASET and SMOKE["n_assets"] == EXPECTED_ASSETS
assert SMOKE["model_smoke"]["gradients_finite"] and np.isfinite(SMOKE["model_smoke"]["loss"])
assert SMOKE["model_smoke"]["prediction_shape"] == [1, SMOKE["n_assets"]]
assert SMOKE["history_shape"][1:] == [cfg.data.lookback, SMOKE["n_assets"]]
assert SMOKE["target_shape"][1:] == [cfg.data.horizon, SMOKE["n_assets"]]
assert SMOKE["max_gap_days"] == cfg.data.max_gap_days
display(pd.DataFrame([{
    "Dataset": SMOKE["dataset"], "Assets": SMOKE["n_assets"],
    "Train windows": SMOKE["train_samples"], "Validation windows": SMOKE["validation_samples"],
    "Test windows (metadata only)": SMOKE["test_samples"],
    "Smoke loss (not a forecast result)": SMOKE["model_smoke"]["loss"],
}]))


In [ ]:
provenance = {
    "git_commit": GIT_SHA, "config": CONFIG, "config_path": CONFIG_PATH,
    "dataset": cfg.data.dataset, "model": cfg.model.history_encoder, "seed": cfg.seed,
    "columns": SMOKE["columns"],
    "sample_counts": {key: SMOKE[key] for key in ("train_samples", "validation_samples", "test_samples")},
    "archives": {path.name: sha256(path) for path in ARCHIVES},
}
provenance_path = RUN_ROOT / "provenance.json"
if provenance_path.exists():
    previous = json.loads(provenance_path.read_text())
    if previous != provenance:
        raise RuntimeError("Experiment provenance changed. Pin REVISION to the recorded commit and restore its archives/config, or choose a new RUN_LABEL.")
else:
    if any((folder / "last.pt").exists() for folder in (PILOT_DIR, FULL_DIR)):
        raise RuntimeError("Existing checkpoints lack provenance; do not adopt them into this experiment.")
    provenance_path.write_text(json.dumps(provenance, indent=2) + "\n")
(RUN_ROOT / "inputs").mkdir(exist_ok=True)
for path in ARCHIVES:
    snapshot = RUN_ROOT / "inputs" / path.name
    if snapshot.exists():
        assert sha256(snapshot) == sha256(path), "Archived source changed"
    else:
        shutil.copy2(path, snapshot)
with (SMOKE_DIR / "pip_freeze.txt").open("w") as stream:
    subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=stream, check=True)
SMOKE_PASSED = True
print("Smoke and provenance checks passed; source archives preserved with this experiment.")


In [ ]:
def inspect_run(folder):
    """Inspect CLI artifacts; last.pt is authoritative after an interruption."""
    folder = Path(folder)
    if (folder / "config.yaml").exists():
        assert json.loads(json.dumps(asdict(load_config(folder / "config.yaml")))) == CONFIG, "Saved config mismatch"
    if not (folder / "last.pt").exists():
        if any((folder / name).exists() for name in ("best.pt", "history.csv", "summary.json")):
            raise RuntimeError(f"{folder}: incomplete artifact set without last.pt; refusing to restart.")
        return None
    for name in ("config.yaml", "standardizer.npz", "best.pt"):
        if not (folder / name).is_file():
            raise FileNotFoundError(folder / name)
    scaler = load_standardizer(folder / "standardizer.npz")
    assert list(scaler.columns) == SMOKE["columns"], "Saved asset names/order mismatch"
    np.testing.assert_array_equal(scaler.mean, SMOKE["standardizer"]["mean"])
    np.testing.assert_array_equal(scaler.std, SMOKE["standardizer"]["std"])
    last = torch.load(folder / "last.pt", map_location="cpu", weights_only=False)
    best = torch.load(folder / "best.pt", map_location="cpu", weights_only=False)
    assert {"model_state_dict", "optimizer_state_dict", "history", "best_epoch", "best_val_loss",
            "torch_rng_state", "cuda_rng_state", "loader_rng_state"} <= last.keys(), "Invalid resume checkpoint"
    assert "model_state_dict" in best, "Invalid best checkpoint"
    history = pd.DataFrame(last["history"])[["epoch", "train_loss", "val_loss"]]
    epoch = int(last["epoch"])
    assert history["epoch"].tolist() == list(range(1, epoch + 1)), "Invalid checkpoint history"
    assert np.isfinite(history.to_numpy()).all(), "Non-finite recorded losses"
    assert best["epoch"] == last["best_epoch"] <= epoch <= cfg.training.epochs
    assert np.isclose(best["val_loss"], last["best_val_loss"], rtol=0, atol=1e-8)
    complete = (folder / "summary.json").exists()
    if complete:
        summary = json.loads((folder / "summary.json").read_text())
        assert epoch == cfg.training.epochs, "Partial run incorrectly marked complete"
        pd.testing.assert_frame_equal(pd.read_csv(folder / "history.csv"), history,
                                      check_dtype=False, rtol=1e-12, atol=1e-12)
        assert summary["best_epoch"] == best["epoch"]
        assert np.isclose(summary["best_val_loss"], best["val_loss"], rtol=0, atol=1e-8)
        assert summary["train_samples"] == SMOKE["train_samples"]
        assert summary["val_samples"] == SMOKE["validation_samples"]
    return {"epoch": epoch, "complete": complete, "history": history}


In [ ]:
def run_training(folder, stop_after_epoch=None):
    assert SMOKE_PASSED, "Run successful smoke/provenance checks first"
    assert json.loads(json.dumps(asdict(load_config(CONFIG_PATH)))) == CONFIG, "Active config changed after smoke"
    assert all(sha256(path) == provenance["archives"][path.name] for path in ARCHIVES), "Source changed after smoke"
    state = inspect_run(folder)
    if state and (state["complete"] or (stop_after_epoch is not None and state["epoch"] >= stop_after_epoch)):
        print("Already reached the requested stage:", folder, "epoch:", state["epoch"])
        return state
    folder.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    command = [sys.executable, "-u", "scripts/train_diffusion.py", "--config", CONFIG_PATH,
               "--output-dir", str(folder), "--device", str(device), "--resume"]
    if stop_after_epoch is not None:
        command += ["--stop-after-epoch", str(stop_after_epoch)]
    # GNU time records this command's CPU peak RSS; GPU snapshots are device-wide,
    # not a claimed per-process peak. No training wrapper or model code is added.
    if Path("/usr/bin/time").exists():
        command = ["/usr/bin/time", "-v", "-o", str(folder / f"resources_{stamp}.txt"), *command]
    session = {"git_commit": GIT_SHA, "stage": folder.name, "seed": cfg.seed,
               "device": str(device), "gpu": GPU_NAME, "torch_version": str(torch.__version__),
               "command": command, "started_utc": stamp, "status": "interrupted"}
    if shutil.which("nvidia-smi"):
        session["gpu_snapshot_before"] = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv"], text=True)
    started = time.perf_counter()
    try:
        run_logged(command, folder / f"training_{stamp}.log")
        session["status"] = "returned"
    except Exception as error:
        session.update(status="failed", error=repr(error))
        raise
    finally:
        session["duration_seconds"] = time.perf_counter() - started
        (folder / f"session_{stamp}.json").write_text(json.dumps(session, indent=2) + "\n")
    state = inspect_run(folder)
    assert state is not None, "Training returned without a resumable checkpoint"
    display(state["history"])
    print("Duration:", session["duration_seconds"], "seconds; device:", device, "GPU:", GPU_NAME)
    print("Checkpoints:", {name: (folder / name).exists() for name in ("last.pt", "best.pt", "history.csv", "summary.json")})
    return state


## D. Short training pilot — explicitly opt in

Select `MODE="pilot"` at the top and run from setup. This invokes the existing CLI
with `--resume --stop-after-epoch 2`, retaining the 50-epoch config. It creates
`pilot/last.pt`, `best.pt`, `history.csv`, `config.yaml` and `standardizer.npz`.
`summary.json` must remain absent. Re-running at epoch 2 does not add two epochs.
Live epoch losses, a log, session JSON, duration/device/GPU and (where available)
GNU-time peak CPU RSS are recorded. GPU memory snapshots are not per-process peaks.
Pilot losses assess numerical/practical feasibility, not forecasting quality.


In [ ]:
if MODE == "pilot":
    pilot_state = run_training(PILOT_DIR, PILOT_EPOCHS)
    assert not pilot_state["complete"], "Pilot must not be labelled a full experiment"
else:
    print("Pilot skipped; select MODE='pilot' to opt in.")


## E. Full experiment — explicitly opt in

After a successful pilot, select `MODE="full"`. The separate `full/` run uses the
original 50-epoch config. Always passing `--resume`, even initially, makes the CLI
save `last.pt` with optimizer and RNG state. `best.pt` is the validation-selected
model. `history.csv` is written on a clean CLI return; after a session interruption
it may lag `last.pt`, whose embedded history is authoritative for resuming.

For session limits, set `FULL_STOP_AFTER_EPOCH` to an **absolute** epoch ceiling,
back up the returned chunk, then restore and raise/remove that ceiling next time.
Partial chunks have no `summary.json` and cannot trigger evaluation. If interrupted
after epoch 50 but before final bookkeeping, resume once to finalize the summary.
A completed, consistent run is skipped. Missing/mixed checkpoints or changed
config/source/Git provenance fail instead of silently restarting. Use the recorded
`REVISION` to resume unchanged code. CUDA reproducibility follows the existing
training/backend behavior; cross-GPU bitwise equivalence is not assumed.


In [ ]:
FULL_COMPLETE = False
if MODE == "full":
    if not (FULL_DIR / "last.pt").exists():
        pilot_state = inspect_run(PILOT_DIR)
        if pilot_state is None or pilot_state["epoch"] < PILOT_EPOCHS:
            raise RuntimeError("Complete the separate pilot before starting a new full run.")
    full_state = run_training(FULL_DIR, FULL_STOP_AFTER_EPOCH)
    FULL_COMPLETE = full_state["complete"]
    if not FULL_COMPLETE:
        print("Full run is partial: back up last.pt and resume this same directory later.")
else:
    print("Full training skipped; select MODE='full' to opt in.")


## F. Completed full run: validation evaluation and reporting

Only a verified complete full run invokes `scripts/evaluate_diffusion.py` with
`--split validation`, loading its saved best-validation checkpoint and scaler.
Existing matching metrics are reused; mismatched metadata fails. Scenario count
and seed remain configured, and batch size is matched across the paired notebooks.
The tables and curves read actual saved outputs; no probabilistic metric is
reimplemented. They are saved as CSV/PNG for later comparison. There is no automatic
conclusion, and no dependency on another experiment having finished. FF100 analysis
remains exploratory/scalability-oriented even if training completes successfully.


In [ ]:
if MODE == "full" and FULL_COMPLETE:
    assert SMOKE_PASSED and inspect_run(FULL_DIR)["complete"]
    metrics_path = FULL_DIR / "validation_probabilistic_metrics.json"
    if not metrics_path.exists() or not (FULL_DIR / "validation_scenarios.npz").is_file():
        run_logged([sys.executable, "scripts/evaluate_diffusion.py", "--run-dir", str(FULL_DIR),
                    "--split", "validation", "--batch-size", str(EVAL_BATCH_SIZE),
                    "--device", str(device)], FULL_DIR / "validation_evaluation.log")
    assert (FULL_DIR / "validation_scenarios.npz").is_file(), "Missing validation scenarios"
    metrics = json.loads(metrics_path.read_text())
    summary = json.loads((FULL_DIR / "summary.json").read_text())
    assert metrics["split"] == "validation" and metrics["checkpoint_epoch"] == summary["best_epoch"]
    assert np.isclose(metrics["checkpoint_val_loss"], summary["best_val_loss"], rtol=0, atol=1e-8)
    assert metrics["n_validation"] == SMOKE["validation_samples"]
    assert metrics["n_scenarios"] == cfg.evaluation.n_scenarios
    assert metrics["sampling_seed"] == cfg.training.validation_seed
    assert metrics["sampling_batch_size"] == EVAL_BATCH_SIZE, "Stored metrics use a different evaluation batch size"
    assert np.isfinite([metrics["crps_mean"], metrics["energy_score"]]).all()
    calibration = pd.DataFrame(metrics["calibration"])
    assert np.isfinite(calibration.to_numpy()).all()
    table = pd.DataFrame([{
        "Dataset": cfg.data.dataset, "Encoder": cfg.model.history_encoder, "Seed": cfg.seed,
        "Best epoch": summary["best_epoch"], "Validation loss": summary["best_val_loss"],
        "CRPS": metrics["crps_mean"], "Energy Score": metrics["energy_score"],
        "Parameters": summary["parameter_count"], "Validation windows": metrics["n_validation"],
        "Git commit": GIT_SHA,
    }])
    display(table)
    display(calibration)
    table.to_csv(FULL_DIR / "validation_report.csv", index=False)
    calibration.to_csv(FULL_DIR / "validation_calibration.csv", index=False)
    history = pd.read_csv(FULL_DIR / "history.csv")
    axes = history.plot(x="epoch", y=["train_loss", "val_loss"], title=f"{EXPERIMENT}: full training")
    axes.set_ylabel("Diffusion loss")
    axes.figure.savefig(FULL_DIR / "loss_curves.png", bbox_inches="tight")
    plt.show()
else:
    print("No full validation evaluation: smoke/pilot/partial histories are not forecasting evidence.")


## G. Opt-in backup, including intermediate checkpoints

Set `UPLOAD_ARTIFACTS=True` and select `UPLOAD_STAGE` before rerunning the final
cell. It can save any valid completed epoch, including an interrupted run with a
valid `last.pt`; a partial/mismatched checkpoint set fails. The selected stage plus
provenance, smoke files and raw archives go under the printed namespace in
`amirentezari/diffusion-portfolio`. Downloading the latest central dataset must
succeed before staging. Only this stage is replaced; the other stage, other models,
and every historical KF12 artifact are preserved. A newer/conflicting remote
checkpoint is never overwritten. Upload errors are raised and local files remain.
Serialize uploads across notebooks: snapshot merging is not a concurrent-write lock.

Also use Kaggle's output download/save-version facilities for urgent manual saves.
This cell cannot run concurrently with a blocking training cell: prefer clean
epoch chunks, or stop training and validate the last completed checkpoint first.


In [ ]:
if not UPLOAD_ARTIFACTS:
    print("Upload disabled. To save a completed epoch before session expiry, set UPLOAD_ARTIFACTS=True,")
    print("select UPLOAD_STAGE='pilot' or 'full', and rerun this cell after the training process stops.")
else:
    assert SMOKE_PASSED and UPLOAD_STAGE in {"pilot", "full"}
    source = RUN_ROOT / UPLOAD_STAGE
    state = inspect_run(source)
    if state is None:
        raise RuntimeError("No valid checkpoint to back up; refusing an empty upload.")
    assert json.loads((RUN_ROOT / "provenance.json").read_text()) == provenance
    for archive in ARCHIVES:
        assert sha256(RUN_ROOT / "inputs" / archive.name) == provenance["archives"][archive.name]
    try:
        # Refresh BEFORE staging. Download failure never creates an empty replacement.
        latest = Path(kagglehub.dataset_download(ARTIFACT_DATASET, force_download=True))
        if not latest.is_dir() or not any(p.is_file() and p.name != ".complete" for p in latest.rglob("*")):
            raise RuntimeError("Central artifact snapshot is empty or missing; refusing replacement.")
        with tempfile.TemporaryDirectory(prefix="high_dimensional_backup_", dir="/kaggle/working") as temporary:
            staging = Path(temporary) / "dataset"
            shutil.copytree(latest, staging)
            namespace = staging / ARTIFACT_NAMESPACE
            if namespace.exists():
                assert (namespace / "provenance.json").is_file(), "Remote namespace lacks provenance"
                assert json.loads((namespace / "provenance.json").read_text()) == provenance, "Remote experiment provenance mismatch"
            destination = namespace / UPLOAD_STAGE
            remote_state = inspect_run(destination)
            if remote_state:
                assert remote_state["epoch"] <= state["epoch"], "Remote checkpoint is newer; refusing to replace it"
                if remote_state["epoch"] == state["epoch"]:
                    assert sha256(destination / "last.pt") == sha256(source / "last.pt"), "Conflicting checkpoint at the same epoch"
            namespace.mkdir(parents=True, exist_ok=True)
            shutil.copy2(RUN_ROOT / "provenance.json", namespace / "provenance.json")
            for name in ("inputs", "smoke"):
                shutil.copytree(RUN_ROOT / name, namespace / name, dirs_exist_ok=True)
            if destination.exists():
                shutil.rmtree(destination)
            shutil.copytree(source, destination)
            assert (destination / "last.pt").is_file() and (destination / "best.pt").is_file()
            kagglehub.dataset_upload(ARTIFACT_DATASET, str(staging),
                version_notes=f"{ARTIFACT_NAMESPACE}/{UPLOAD_STAGE}: epoch {state['epoch']}",
                ignore_patterns=[".complete"])
        print("Uploaded:", ARTIFACT_NAMESPACE, UPLOAD_STAGE, "epoch:", state["epoch"])
    except Exception as error:
        raise RuntimeError("Backup failed; local artifacts remain available in RUN_ROOT.") from error
